In [ ]:
import pandas as pd
import numpy as np

In [ ]:
lookup = (
    pd.read_parquet('../data/mb_album_artists.parquet', columns=['album_id', 'album_name', 'artist_name'])
    .drop_duplicates(subset='album_id')
    .set_index('album_id')
)

print(f"Lookup table: {len(lookup):,} albums")
print(f"Missing artist_name: {lookup['artist_name'].isna().sum():,}")
print(f"Missing album_name : {lookup['album_name'].isna().sum():,}")
lookup.head()

In [ ]:
def search_artist(name, max_results=10):
    """Case-insensitive partial match on artist_name. Returns album_id, album_name, artist_name."""
    mask = lookup['artist_name'].str.contains(name, case=False, na=False)
    return lookup[mask].head(max_results)

search_artist("Massive Attack")

In [ ]:
import joblib
from scipy.sparse import load_npz

model                = joblib.load('../data/model/knn_model.joblib')
X_knn_norm           = load_npz('../data/model/X_knn_norm.npz')
album_ids_annotated  = np.load('../data/model/album_ids_annotated.npy', allow_pickle=True)
has_features         = np.load('../data/model/has_features.npy')

# Index for O(1) album_id → row lookup in the fitted model
album_id_to_row = {aid: i for i, aid in enumerate(album_ids_annotated)}

print(f"Model loaded: {X_knn_norm.shape[0]:,} albums x {X_knn_norm.shape[1]:,} features")

In [ ]:
def recommend(album_id, n=10):
    """
    Return the n most similar albums to album_id, excluding all albums by the
    same artist. Returns a DataFrame with album_name and artist_name, or a
    message if the album has no features.
    """
    if album_id not in album_id_to_row:
        return f"No recommendations: album {album_id} has no feature data."

    # Identify the input artist so we can exclude their other albums
    input_artist = lookup.loc[album_id, 'artist_name'] if album_id in lookup.index else None

    # Fetch extra candidates to absorb same-artist exclusions
    row = album_id_to_row[album_id]
    distances, indices = model.kneighbors(X_knn_norm[row], n_neighbors=n * 5)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        aid = album_ids_annotated[idx]
        if aid == album_id:
            continue
        row_data = lookup.loc[aid] if aid in lookup.index else {'album_name': None, 'artist_name': None}
        if input_artist and row_data['artist_name'] == input_artist:
            continue
        results.append({
            'album_id':    aid,
            'album_name':  row_data['album_name'],
            'artist_name': row_data['artist_name'],
            'distance':    round(dist, 4),
        })
        if len(results) == n:
            break

    return pd.DataFrame(results)

In [ ]:
# Smoke test — find Massive Attack albums and recommend from one
results = search_artist("Massive Attack")
print(results.to_string())
print()

album_id = results.index[0]
recommend(album_id)